# GEBCO AGOL
Acessing GEBCO data via its ArcGIS Online Resource. 

GEBCO Bathymetry is available as an AGOL Image Collection service via ESRI's Living Atlas:  
<https://www.arcgis.com/home/item.html?id=97d2ea583d3146e0ba72bc498a9c571f>

It's REST endpoint is here:  
<https://di-elevation.img.arcgis.com/arcgis/rest/services/gebco/ImageServer>

This notebook demonstrates how to use this resource in an analytical context. Specifically, we define an area of intersted (a polygon) and extract GEBCO data within that area. The polygons we'll use here is [ESRI's Global H3 Hexbins Levels 0 to 6](https://dukeuniv.maps.arcgis.com/home/item.html?id=5334817d9f6a41ae88ca6b4182fc07dc) dataset. 

**CONDA RECIPE**  
This notebook uses the default `arcgispro-py3` Conda environment. 

## Set up

In [ ]:
# Import packages
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from arcgis.raster import ImageryLayer, Raster
from arcgis.geometry import Polygon, filters, project
from arcgis.raster.functions import clip, greater_than

In [ ]:
# Connect to AGOL via our ArcGIS Pro token (required for H3 access)
gis = GIS('home')

## Fetch the H3 hexagon dataset as a feature service
The H3 feature service layer is hosted here:  
<https://www.arcgis.com/home/item.html?id=8e1e15e336024f01894a6253bbf756d7>

We'll use the datasets ItemID (in its URL) to fetch the feature service item and its layers. 

In [ ]:
# Get the H3 item
h3_items = gis.content.get('5334817d9f6a41ae88ca6b4182fc07dc')
h3_items.layers

Inspection of the info on this feature service reveals it has layers for each level in the H3 dataset. We'll chose level 4 for our analysis, which is the 3rd item in the list of layers. 

In [ ]:
#Fetch the level 4 h3 hexagons (the 3rd item in the list of layers)
h3_level4_lyr = h3_items.layers[2]

Now we'll select a single hexagon feature for our analysis. Hexagon with a GRID_ID of `842ad83ffffffff` falls off the coast of NC. 

In [ ]:
the_hexagon_sdf = h3_level4_lyr.query("GRID_ID = '842ad83ffffffff'").sdf
the_hexagon_sdf

In [ ]:
the_hexagon = the_hexagon_sdf.loc[0,'SHAPE']
the_hexagon

## Fetch the GEBCO image service
The GEBCO data is hosted as an *imagery service*. 
* Documentation: https://developers.arcgis.com/python/latest/api-reference/arcgis.raster.toc.html#arcgis.raster.ImageryLayer

In [ ]:
# Fetch the service using its REST endpoint URL
bathy_lyr = ImageryLayer(url = 'https://di-elevation.img.arcgis.com/arcgis/rest/services/gebco/ImageServer')

# Confirm its type
type(bathy_lyr)

## Analysis
1. Project the hexagon to match the spatial reference of the GEBCO bathymetry layer
1. Clip the bathymetry layer by the hexagon.
1. Convert the clipped bathymetry to a binary layer of pixels meeting the depth threshold.

In [ ]:
#Project the hexagon to match the raster's spatial reference
the_hexagon_prj = project(
    geometries = [the_hexagon],
    in_sr = the_hexagon.spatialReference,
    out_sr = bathy_lyr.properties.spatialReference,
    gis = gis
)[0]

#Display
the_hexagon_prj

#### Compute stats of the pixels inside the polygon.

In [ ]:
#Create a Raster object from the bathymetry layer
bathy_raster = Raster(bathy_lyr.url)

#Execute the `summarize` command
hex_stats = bathy_raster.summarize(geometry=the_hexagon_prj)

#Show the results
hex_stats

### Clip the raster to the hexagon
https://developers.arcgis.com/python/latest/api-reference/arcgis.raster.functions.html#clip

In [ ]:
#Clip the GEBCO bathymetry layer by the hexagon
clipped_bathy = clip(
    raster = bathy_lyr,
    geometry = the_hexagon_prj,
    clip_outside = True
)

### Use Map Algebra to convert to a binary raster

In [ ]:
#Threshold to pixels between 1000 and 500 m of depth
min_depth = 500
max_depth = 1000

binary_result = (clipped_bathy > max_depth*-1) & (clipped_bathy < min_depth*-1)

### Compute stats from the binary dataset  
<https://developers.arcgis.com/python/latest/api-reference/arcgis.raster.toc.html#arcgis.raster.ImageryLayer.compute_stats_and_histograms>

In [ ]:
the_stats = binary_result.compute_stats_and_histograms(
    geometry=the_hexagon_prj
)['statistics'][0]
the_stats


In [ ]:
#Report the output
print(f'{the_stats['mean']*100:.2f}% of pixels are between {min_depth} and {max_depth} m below surface')

### Export the raster locally

In [ ]:
# Define the output folder
from pathlib import Path
output_folder = Path.cwd().parent / 'data' / 'processed'
output_folder.mkdir(parents=True, exist_ok=True)

Compute the output dimensions

In [ ]:
# Original cell size
cell_x = abs(bathy_lyr.properties.pixelSizeX)
cell_y = abs(bathy_lyr.properties.pixelSizeY)

# Get the polygon's bounding extent
xmin, ymin, xmax, ymax = the_hexagon_prj.extent

# Required dimensions
from math import ceil
cols = ceil((xmax - xmin) / cell_x)
rows = ceil((ymax - ymin) / cell_y)

print(f"Output dimensions: {rows} rows x {cols} columns")

In [ ]:
# Get the polygon's bounding extent
xmin, ymin, xmax, ymax = the_hexagon_prj.extent

In [ ]:
# Export the classified raster
output_file = binary_result.export_image(
    bbox=f"{xmin},{ymin},{xmax},{ymax}",
    bbox_sr=the_hexagon_prj["spatialReference"],
    image_sr=the_hexagon_prj["spatialReference"],
    size=[cols, rows],
    export_format="tiff",
    pixel_type="U8",
    interpolation="RSP_NearestNeighbor",
    compression="LZ77",
    f="image",
    save_folder=str(output_folder),
    save_file="binary_classification2.tif"
)

print(output_file)